In [1]:
!nohup vllm serve --model "google/gemma-4-E2B-it" --enable-auto-tool-choice --tool-call-parser gemma4 > vllm.log 2>&1 &

In [9]:
from dataclasses import dataclass
from pydantic_ai import Agent, ModelRetry, RunContext, Tool
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.vllm import VLLMProvider
from pydantic_ai.messages import ModelMessage

In [19]:
model_name = "google/gemma-4-E2B-it"
base_url = "http://localhost:8000/v1"

model = OpenAIChatModel(
model_name,
provider=VLLMProvider(base_url=base_url))


agent = Agent(model,
              system_prompt="You are a customer support agent. You can process refunds, but you must use the process_refund tool to do so."
)

In [20]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.messages import ModelMessage



@agent.tool
async def bixby_code(ctx: RunContext,number: int) -> str:
    "Calculates bixby code"
    return str(10 * number + 55) + "A" * (number%2) + "B" * ((number+1) %2)

# 2. Define the tool requiring human validation
@agent.tool
def process_refund(ctx: RunContext, amount: float, customer_name: str) -> str:
    print(f"\n[HUMAN IN THE LOOP REQUIRED]")
    print(f"The agent is attempting to refund ${amount} to {customer_name}.")

    # Pause execution and wait for human input
    while True:
        decision = input("Type ACCEPT to authorize or DECLINE to reject: ").strip().upper()
        if decision == "ACCEPT":
            print("[SYSTEM] Refund authorized.\n")
            # The LLM receives this success message
            return f"Success: Refund of ${amount} processed."
        elif decision == "DECLINE":
            print("[SYSTEM] Refund blocked.\n")
            # The LLM receives this rejection message and can adapt its response
            return "Error: Human supervisor declined this action. Inform the user."
        else:
            print("Invalid input. Please type ACCEPT or DECLINE.")

In [24]:
chat_history: list[ModelMessage] = []

while True:
    user_input = input("\nUser: ")
    if user_input.lower() in ['exit', 'quit']:
        break

    # Pass the accumulated history into the run
    result = await agent.run(user_input, message_history=chat_history)

    print(f"Agent: {result.output}")

    # Update the history with the latest turn (includes the user prompt, tool calls, and final response)
    chat_history = result.all_messages()


User: Hi
Agent: Hello! I'm here to help you with your request. Are you looking to process a refund, or do you have another question?

User: What is the bixby number for 1
Agent: The bixby code for 1 is 65A.

User: can I get a refund?
Agent: I can certainly help you with a refund. To process a refund, I will need two pieces of information:

1.  **The amount** you wish to be refunded.
2.  **The customer's name**.

Could you please provide me with those details?

User: 10 euros for Suzie

[HUMAN IN THE LOOP REQUIRED]
The agent is attempting to refund $10.0 to Suzie.
Type ACCEPT to authorize or DECLINE to reject: DECLINE
[SYSTEM] Refund blocked.

Agent: I have attempted to process a refund of 10 euros for Suzie.

However, I received an error stating: **"Error: Human supervisor declined this action. Inform the user."**

This means that, for some reason, the refund could not be processed automatically. I recommend contacting a human supervisor or another support channel to discuss this furt

In [25]:
chat_history: list[ModelMessage] = []

while True:
    user_input = input("\nUser: ")
    if user_input.lower() in ['exit', 'quit']:
        break

    # Pass the accumulated history into the run
    result = await agent.run(user_input, message_history=chat_history)

    print(f"Agent: {result.output}")

    # Update the history with the latest turn (includes the user prompt, tool calls, and final response)
    chat_history = result.all_messages()


User: hi
Agent: Hi! How can I help you today? Are you looking to process a refund?

User: yes
Agent: I can certainly help you with a refund. To proceed, I will need two pieces of information:

1.  **The amount** you wish to be refunded.
2.  **The customer's name**.

Please provide me with those details.

User: 10 euro for Jill

[HUMAN IN THE LOOP REQUIRED]
The agent is attempting to refund $10.0 to Jill.
Type ACCEPT to authorize or DECLINE to reject: asda
Invalid input. Please type ACCEPT or DECLINE.
Type ACCEPT to authorize or DECLINE to reject: adas
Invalid input. Please type ACCEPT or DECLINE.
Type ACCEPT to authorize or DECLINE to reject: asdasd
Invalid input. Please type ACCEPT or DECLINE.
Type ACCEPT to authorize or DECLINE to reject: ACCEPT
[SYSTEM] Refund authorized.

Agent: Your refund of 10 euro for Jill has been successfully processed. Is there anything else I can assist you with today?

User: I want to make a refund again. This time it will be double the amount of the prev